# Задача 3 – AVL

## Код

In [1]:
class Node:
    def __init__(self, val):
        self.val = val
        self.left = None
        self.right = None
        self.height = 1


class AVL:
    def __init__(self):
        self.root = None

    def _height(self, node):
        return 0 if node is None else node.height

    def _update(self, node):
        node.height = 1 + max(self._height(node.left), self._height(node.right))

    def _balance(self, node):
        return self._height(node.left) - self._height(node.right)

    def _rotate_right(self, node):
        new_root = node.left
        node.left = new_root.right
        new_root.right = node
        self._update(node)
        self._update(new_root)
        return new_root

    def _rotate_left(self, node):
        new_root = node.right
        node.right = new_root.left
        new_root.left = node
        self._update(node)
        self._update(new_root)
        return new_root

    def _rebalance(self, node):
        self._update(node)
        balance = self._balance(node)
        if balance > 1:
            if self._balance(node.left) < 0:
                node.left = self._rotate_left(node.left)
            return self._rotate_right(node)
        if balance < -1:
            if self._balance(node.right) > 0:
                node.right = self._rotate_right(node.right)
            return self._rotate_left(node)
        return node

    def insert(self, val):
        self.root = self._insert(self.root, val)

    def _insert(self, node, val):
        if node is None:
            return Node(val)
        if val < node.val:
            node.left = self._insert(node.left, val)
        elif val > node.val:
            node.right = self._insert(node.right, val)
        else:
            return node
        return self._rebalance(node)

    def delete(self, val):
        self.root = self._delete(self.root, val)

    def _delete(self, node, val):
        if node is None:
            return None
        if val < node.val:
            node.left = self._delete(node.left, val)
        elif val > node.val:
            node.right = self._delete(node.right, val)
        else:
            if node.left is None:
                return node.right
            if node.right is None:
                return node.left
            next_node = node.right
            while next_node.left is not None:
                next_node = next_node.left
            node.val = next_node.val
            node.right = self._delete(node.right, next_node.val)
        return self._rebalance(node)

    def search(self, val):
        def find(node):
            if node is None:
                return False
            if val == node.val:
                return True
            if val < node.val:
                return find(node.left)
            return find(node.right)

        return find(self.root)

## Идея

Для дерева заводим класс Node со значением val, ссылками left и right и высотой height. В классе AVL храним корень root. Одинаковые значения повторно не добавляем, поиск возвращает True или False

Вставку и поиск выполняем как в обычном BST: меньшее значение ищем слева, большее справа. При удалении узел без детей заменяем на None, с одним ребенком — на этого ребенка. Если детей два, записываем минимальное значение из правого поддерева и удаляем его оттуда

После вставки и удаления при возврате из рекурсии обновляем высоту и проверяем разницу высот. Если слева выше больше чем на 1, делаем правый поворот, если справа — левый. Когда тяжелое поддерево наклонено в другую сторону, сначала поворачиваем его, потом сам узел

Повороты сохраняют порядок BST, тк промежуточное поддерево переносим между теми же соседними значениями. После них пересчитываем высоты снизу вверх. Таким образом, при возврате каждый узел снова сбалансирован. Удаление отсутствующего значения ничего не меняет

Асимптотическая сложность вставки, удаления и поиска: **O(log n)**, где n — количество узлов. Высота AVL-дерева O(log n), на каждом уровне выполняем постоянное количество действий и поворотов

Объем дополнительной памяти: **O(log n)** на рекурсию. Само дерево занимает **O(n)**

## Визуализация

![Схема](schema.png)

Удаляем 5 из дерева с корнем 4. После удаления справа высота 0, слева 2, поэтому делаем правый поворот. Получаем корень 2, высоту 3 и баланс -1

| Шаг удаления 5 | Действие |
| --- | --- |
| 1 | 5 > 4, идем вправо |
| 2 | удаляем лист 5, возвращаем None |
| 3 | у 4 баланс 2, у его левого ребенка баланс 0 |
| 4 | правый поворот вокруг 4, новый корень 2 |

## Тесты

In [2]:
def check_tree(tree, expected):
    values = []

    def check(node):
        if node is None:
            return 0
        left = check(node.left)
        values.append(node.val)
        right = check(node.right)
        assert abs(left - right) <= 1
        assert node.height == 1 + max(left, right)
        return node.height

    check(tree.root)
    assert values == sorted(expected)


def make_tree(values):
    tree = AVL()
    for val in values:
        tree.insert(val)
    return tree


def test_rotations():
    for values in ([3, 2, 1], [1, 2, 3], [3, 1, 2], [1, 3, 2]):
        tree = AVL()
        seen = set()
        for val in values:
            tree.insert(val)
            seen.add(val)
            check_tree(tree, seen)
        assert tree.root.val == 2
    print('OK')


def test_empty_and_duplicates():
    tree = AVL()
    assert tree.search(1) is False
    tree.delete(1)
    check_tree(tree, [])
    tree.insert(0)
    tree.insert(0)
    assert tree.search(0) is True
    check_tree(tree, [0])
    tree.delete(0)
    assert tree.root is None
    print('OK')


def test_delete_cases():
    for values, target in [([2, 1, 3], 1), ([2, 1], 2),
                           ([2, 3], 2), ([4, 2, 6, 1, 3, 5, 7], 4),
                           ([4, 2, 5, 1, 3], 5), ([2, 1, 4, 3, 5], 1),
                           ([5, 2, 8, 3], 8), ([5, 2, 8, 7], 2)]:
        tree = make_tree(values)
        tree.delete(target)
        check_tree(tree, set(values) - {target})
        assert tree.search(target) is False
    print('OK')


def test_sequence():
    tree = AVL()
    for val in range(20):
        tree.insert(val)
        check_tree(tree, range(val + 1))
    for val in range(20):
        assert tree.search(val) is True
        tree.delete(val)
        check_tree(tree, range(val + 1, 20))
    assert tree.root is None
    print('OK')


test_rotations()
test_empty_and_duplicates()
test_delete_cases()
test_sequence()

OK
OK
OK
OK
